# 08c — Exposure-aware policy experiment

Stage 08b showed two different exposure-related failure modes:

- SPARSE users can be blocked by the absolute min_dates = 3 requirement;
- DENSE/mobile users can have enough support but lower absolute share/margin because semantic mass is distributed across more locations.

Stage 08c compares predeclared exposure-aware candidate rules against the frozen production baseline.

Production remains HOME = 27 and OFFICE = 16 throughout this notebook.

No rule here silently changes production. New candidates remain experimental/tiered until robustness is reviewed.

In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

import pandas as pd

REPO_URL='https://github.com/tanh1c/geolife.git'
BRANCH=os.environ.get('GEOLIFE_REPO_BRANCH','main')
REPO_DIR=Path(os.environ.get('GEOLIFE_REPO_DIR','/root/geolife'))
V=Path('/mnt/geolife-data')
ROOT=V/'cache'/'cp2_v2'
C=ROOT/'08c_exposure_aware_policy'
C.mkdir(parents=True,exist_ok=True)

def run(cmd,cwd=None):
    subprocess.run(cmd,cwd=cwd,check=True)

if not (REPO_DIR/'.git').exists():
    run(['git','clone','--branch',BRANCH,'--single-branch',REPO_URL,str(REPO_DIR)])
else:
    run(['git','-C',str(REPO_DIR),'fetch','origin',BRANCH])
    run(['git','-C',str(REPO_DIR),'checkout',BRANCH])
    run(['git','-C',str(REPO_DIR),'reset','--hard',f'origin/{BRANCH}'])

run([sys.executable,'-m','pip','install','-q','-e','.[dev]'],cwd=REPO_DIR)
for p in (str(REPO_DIR),str(REPO_DIR/'src')):
    if p not in sys.path:
        sys.path.insert(0,p)

from geolife.model import HomeOfficeConfig, build_semantic_locations, infer_home_office

spec=importlib.util.spec_from_file_location(
    's08c',
    REPO_DIR/'analysis'/'08c_exposure_aware_policy.py',
)
assert spec is not None and spec.loader is not None
s08c=importlib.util.module_from_spec(spec)
sys.modules[spec.name]=s08c
spec.loader.exec_module(s08c)

print('sha:',subprocess.run(
    ['git','-C',str(REPO_DIR),'rev-parse','--short','HEAD'],
    capture_output=True,text=True,check=True
).stdout.strip())
print(s08c.synthetic_self_check())

## 1. Frozen baseline + prerequisite caches

08c consumes measured private artifacts from 08b, 08a, 07j and Stage 05. It rebuilds only the frozen production output from the existing stay cache to enforce collision checks.

In [ ]:
def find_stay_cache():
    candidates=[
        V/'cache'/'03a_user_behavior_deep_dive'/'stays_baseline_v1.pkl',
        V/'artifacts'/'03a'/'stays_baseline_v1.pkl',
        REPO_DIR/'artifacts'/'03a'/'stays_baseline_v1.pkl',
    ]
    for p in candidates:
        if p.exists():
            return p
    hits=sorted(V.glob('**/stays_baseline_v1.pkl'))
    if hits:
        return hits[0]
    raise FileNotFoundError('stays_baseline_v1.pkl')

stays=pd.read_pickle(find_stay_cache())
cfg=HomeOfficeConfig()
semantic_stays,locations=build_semantic_locations(stays,config=cfg)
production=infer_home_office(stays,config=cfg)

counts=production['label'].value_counts().to_dict()
assert len(stays)==5821
assert counts.get('HOME',0)==27
assert counts.get('OFFICE',0)==16

paths={
    'profiles':ROOT/'08b_exposure_bias_audit'/'user_exposure_profiles_private.pkl',
    'diagnostics':ROOT/'08b_exposure_bias_audit'/'gate_diagnostics_private.pkl',
    'home_expansion':ROOT/'08a_home_coverage_expansion'/'home_expansion_evidence_private.pkl',
    'office_near_miss':ROOT/'07j_near_miss_office_audit'/'near_miss_office_audit_private.pkl',
    'assignments':ROOT/'05_home_office_reliability_validation'/'assignments_private.pkl',
}
missing=[str(p) for p in paths.values() if not p.exists()]
if missing:
    raise FileNotFoundError('Missing Stage-08c prerequisite caches:\n'+'\n'.join(missing))

profiles=pd.read_pickle(paths['profiles'])
diagnostics=pd.read_pickle(paths['diagnostics'])
home_expansion=pd.read_pickle(paths['home_expansion'])
office_near_miss=pd.read_pickle(paths['office_near_miss'])
assignments=pd.read_pickle(paths['assignments'])

gate=pd.DataFrame([{
    'production_HOME':counts.get('HOME',0),
    'production_OFFICE':counts.get('OFFICE',0),
    'HOME_PROBABLE_08a':int(home_expansion['home_expansion_tier'].eq('HOME_PROBABLE').sum()),
    'OFFICE_near_miss_07j':int(
        office_near_miss.loc[
            office_near_miss['audit_group'].astype(str).ne('baseline'),
            'user_id'
        ].astype(str).nunique()
    ),
    'diagnostic_rows_08b':len(diagnostics),
}])
assert int(gate.iloc[0]['HOME_PROBABLE_08a'])==5
assert int(gate.iloc[0]['OFFICE_near_miss_07j'])==9
print('Stage-08c input gate: PASS')
display(gate)

## 2. Predeclared policy branches

### SPARSE HOME — support-aware

Only relax the absolute date count from 3 to 2, and only when all of these hold:

- SPARSE HOME exposure;
- exactly 2 observed HOME opportunity dates;
- same raw top candidate supported on 2/2 dates;
- baseline HOME share >= 0.50;
- baseline HOME margin >= 0.20;
- recurrence chooses the same location;
- no collision with production OFFICE.

So share and margin are not relaxed for sparse HOME.

### DENSE HOME — concentration-aware

Candidate must already be HOME_PROBABLE from 08a, be the eligible fixed-window top candidate, and fail only through share/margin concentration. Replace absolute share/margin with relative top-vs-runner-up dominance >= 0.625.

### DENSE OFFICE — conservative concentration-aware

Only the nine already-audited 07j near misses are considered. Relative dominance must be >= 0.60, no production-HOME collision, dropout retention >= 0.80, and at least one static/split/heldout corroborator must support candidate identity.

Sparse OFFICE is intentionally not relaxed because 08b found 0/38 selection under fixed, HoWDe and recurrence.

Branch IDs used by the experiment:

- `HOME_SPARSE_2OF2_RECURRENCE`
- `HOME_DENSE_RELATIVE_DOMINANCE`
- `OFFICE_DENSE_ROBUST_RELATIVE`


In [ ]:
print('HOME_RELATIVE_DOMINANCE_FLOOR =',s08c.HOME_RELATIVE_DOMINANCE_FLOOR)
print('OFFICE_RELATIVE_DOMINANCE_FLOOR =',s08c.OFFICE_RELATIVE_DOMINANCE_FLOOR)

## 3. SPARSE HOME support-aware candidates

In [ ]:
sparse_home=s08c.sparse_home_support_candidates(
    diagnostics,
    assignments,
    production,
)

sparse_review=sparse_home[[
    c for c in [
        'user_id',
        'home_opportunity_dates',
        'raw_top_relevant_dates',
        'raw_top_date_coverage',
        'raw_top_relevant_share',
        'raw_top_share_margin',
        'recurrence_matches_raw_top',
        'collides_with_production_office',
        'candidate_location_id',
    ]
    if c in sparse_home.columns
]].copy()
sparse_review=sparse_review.sort_values('user_id',kind='stable').reset_index(drop=True)
sparse_review.insert(0,'audit_id',[f'SH{i:02d}' for i in range(1,len(sparse_review)+1)])
display(sparse_review.drop(columns=['user_id'],errors='ignore'))

## 4. DENSE HOME concentration-aware candidates

This branch asks whether the existing 08a HOME_PROBABLE candidates also satisfy a relative concentration rule derived from the frozen HOME boundary.

In [ ]:
dense_home=s08c.dense_home_concentration_candidates(
    diagnostics,
    home_expansion,
)

dense_home_review=dense_home[[
    c for c in [
        'user_id',
        'home_exposure_regime',
        'gate_status',
        'relevant_dwell_share',
        'share_margin',
        'relative_top2_dominance',
        'external_exact_family_count',
        'candidate_matches_eligible_top',
        'candidate_location_id',
    ]
    if c in dense_home.columns
]].copy()
dense_home_review=dense_home_review.sort_values('user_id',kind='stable').reset_index(drop=True)
dense_home_review.insert(0,'audit_id',[f'DH{i:02d}' for i in range(1,len(dense_home_review)+1)])
display(dense_home_review.drop(columns=['user_id'],errors='ignore'))

## 5. DENSE OFFICE near-miss concentration-aware candidates

First show the relative-dominance candidate pool; then identify the stricter robust subset.

In [ ]:
dense_office=s08c.dense_office_near_miss_candidates(
    diagnostics,
    office_near_miss,
)

dense_office_review=dense_office[[
    c for c in [
        'user_id',
        'audit_group',
        'office_exposure_regime',
        'relevant_dwell_share',
        'share_margin',
        'relative_top2_dominance',
        'static_comparator_match_count',
        'split_full_candidate_match_count',
        'full_candidate_heldout_top1',
        'dropout_candidate_retention',
        'robustness_corroborated',
        'robust_policy_candidate',
        'candidate_location_id',
    ]
    if c in dense_office.columns
]].copy()
dense_office_review=dense_office_review.sort_values(
    ['robust_policy_candidate','user_id'],
    ascending=[False,True],
    kind='stable',
).reset_index(drop=True)
dense_office_review.insert(0,'audit_id',[f'DO{i:02d}' for i in range(1,len(dense_office_review)+1)])
display(dense_office_review.drop(columns=['user_id'],errors='ignore'))

## 6. Candidate funnel + robustness summary

In [ ]:
funnel=s08c.candidate_funnel(
    sparse_home,
    dense_home,
    dense_office,
)
office_robustness=s08c.office_robustness_summary(dense_office)

print('EXPOSURE-AWARE CANDIDATE FUNNEL')
display(funnel)

print('DENSE OFFICE ROBUSTNESS')
display(office_robustness)

## 7. Policy coverage comparison

Coverage counts below are experimental/tiered counts only. Production remains unchanged.

In [ ]:
coverage=s08c.policy_coverage_summary(
    sparse_home,
    dense_home,
    dense_office,
    baseline_home=27,
    baseline_office=16,
)
assert coverage['production_changed'].eq(False).all()

print('POLICY COVERAGE COMPARISON')
display(coverage)

## 8. Overlap diagnostics

The dense HOME rule is intentionally nested inside the previously validated HOME_PROBABLE tier. Sparse HOME is a new support-normalized candidate branch. Dense OFFICE is a filtered subset of the nine already-audited near misses.

In [ ]:
overlap=pd.DataFrame([{
    'HOME_PROBABLE_08a':int(home_expansion['home_expansion_tier'].eq('HOME_PROBABLE').sum()),
    'HOME_dense_relative_pass':int(dense_home['user_id'].nunique()),
    'HOME_sparse_support_new':int(sparse_home['user_id'].nunique()),
    'OFFICE_near_miss_07j':int(
        office_near_miss.loc[
            office_near_miss['audit_group'].astype(str).ne('baseline'),
            'user_id'
        ].astype(str).nunique()
    ),
    'OFFICE_dense_relative_pool':int(dense_office['user_id'].nunique()),
    'OFFICE_dense_robust_pass':int(
        dense_office.loc[dense_office['robust_policy_candidate'].fillna(False),'user_id']
        .astype(str).nunique()
    ),
}])
display(overlap)

## 9. Save Stage-08c outputs

User identities stay private on the Modal Volume. Aggregate policy counts are safe for documentation.

In [ ]:
# Private candidate sets
sparse_home.to_pickle(C/'sparse_home_support_candidates_private.pkl')
dense_home.to_pickle(C/'dense_home_concentration_candidates_private.pkl')
dense_office.to_pickle(C/'dense_office_near_miss_candidates_private.pkl')

# Aggregate
gate.to_csv(C/'input_gate.csv',index=False)
funnel.to_csv(C/'candidate_funnel.csv',index=False)
office_robustness.to_csv(C/'office_robustness_summary.csv',index=False)
coverage.to_csv(C/'policy_coverage_comparison.csv',index=False)
overlap.to_csv(C/'policy_overlap_summary.csv',index=False)

print('saved:',C)

## Interpretation boundary

Stage 08c is a policy experiment, not a production migration.

A useful outcome would be one of:

- sparse HOME recovers a small number of strongly corroborated 2-of-2 cases;
- the relative-dominance rule recovers most/all 08a HOME_PROBABLE without broad over-selection;
- dense OFFICE robust filtering leaves only a small subset of the nine near misses.

A bad outcome would be large candidate expansion with weak corroboration. In that case the adaptive branch should remain research-only.

No result here establishes semantic accuracy or ground truth.